# Exercice 04 – Extraire les données de contrats de leasing

Le dossier `contrats/` contient 20 contrats de leasing automobile (fictifs), en PDF, issus de trois agences différentes. Chaque agence a sa propre mise en page et son propre vocabulaire (*client* ou *preneur*, *mensualité*, *loyer* ou *redevance*…). Certains contrats font deux pages.

**Objectif** : pour chaque contrat, extraire les informations ci-dessous avec un modèle vision et une **sortie structurée** (schéma Pydantic), les rassembler dans un `DataFrame` **polars**, et mesurer la qualité de l'extraction.

**Serveur** : un modèle vision avec son `mmproj` (voir le cours sur les images). Pensez à vérifier `requests.get(f"{BASE_URL}/props").json()["modalities"]`.

## Les informations à extraire

| Champ | Type | Remarque |
|---|---|---|
| `numero_contrat` | texte | ex. `ALW-2026-1914` |
| `date_contrat` | date | au format `AAAA-MM-JJ` quel que soit le format sur le document |
| `agence` | texte | nom de la société de leasing |
| `client` | texte | le client (ou « preneur »), **pas** le conseiller commercial |
| `vehicule.marque`, `vehicule.modele` | texte | |
| `vehicule.carburant` | choix | `essence`, `diesel`, `hybride`, `hybride rechargeable`, `électrique` |
| `duree_mois`, `km_annuel` | entier | |
| `acompte_eur` | nombre | 0 s'il n'y en a pas |
| `mensualite_tvac_eur` | nombre | TVA comprise ; si le montant a été corrigé à la main, la valeur corrigée |
| `assurance.type` | choix | `omnium`, `mini-omnium`, `rc`, `aucune` |
| `assurance.compagnie` | texte | vide si RC seule ou aucune assurance |
| `assurance.prime_mensuelle_eur` | nombre | 0 si aucune prime séparée |
| `signature_client`, `signature_agence` | booléen | `True` seulement si une signature manuscrite est présente |
| `annule` | booléen | `True` si le document porte une mention d'annulation |

## Ouvrir un PDF en Python

On utilise `pymupdf` (`pip install pymupdf`). Un PDF peut contenir :

- **du texte** (PDF « natif », généré par un logiciel) : `page.get_text()` le renvoie directement ;
- **seulement des images** (PDF **scanné**) : `page.get_text()` renvoie une chaîne vide, il faut transformer la page en image et la donner au modèle vision.

Dans ce lot, **certains contrats sont natifs, d'autres sont des scans**. Le code suivant vous montre la différence.

In [ ]:
import base64
import pymupdf
from IPython.display import Image, display

doc = pymupdf.open("contrats/contrat_01.pdf")   # essayez aussi contrat_04.pdf (scan)
print(len(doc), "page(s)")

# 1) Le texte, s'il existe
texte = doc[0].get_text()
print("Scan (pas de texte)" if not texte.strip() else texte[:500])

# 2) La page en image : fonctionne dans tous les cas
pix = doc[0].get_pixmap(dpi=110)       # 110 dpi : lisible, sans exploser le nombre de tokens
png = pix.tobytes("png")
display(Image(png, width=450))

# 3) L'image prête à être envoyée au modèle (comme dans le cours sur les images)
data_uri = "data:image/png;base64," + base64.b64encode(png).decode()

## À faire

1. Écrire les classes Pydantic correspondant au tableau des champs (`Vehicule`, `Assurance`, `Contrat`).
2. Écrire `extraire(chemin_pdf) -> Contrat` qui envoie **toutes les pages** du contrat au modèle, avec `response_format` et le schéma.
3. Boucler sur les 20 contrats, enregistrer les résultats (plus `prompt_tokens` et la durée) dans `resultats_contrats.csv`.
4. Évaluer : le fichier `verite_terrain.json` avec les valeurs attendues vous sera donné en fin de séance. Calculez le taux de bonnes réponses **par champ**, puis séparément pour les contrats **natifs** et **scannés**.
5. Répondez : quels champs posent le plus de problèmes ? Pourquoi ? Qu'est-ce qui change entre un PDF natif et un scan ?

> **Tips**
> - Le system prompt compte autant que le schéma : rappelez les règles du tableau (TVAC, corrections manuscrites, client ≠ conseiller, ce qui compte comme signature).
> - Les `description` de `Field(...)` sont envoyées au modèle avec le schéma : utilisez-les.
> - Une page A4 à 110 dpi coûte environ un millier de tokens : vérifiez que votre `-c` suffit pour un contrat de deux pages.
> - `temperature=0` et vérifiez `finish_reason`.
> - `pl.DataFrame(liste_de_dicts)` puis `.write_csv(...)` ; `pl.read_json("verite_terrain.json")` lit la vérité terrain.
> - Pour les PDF natifs, vous pouvez **aussi** envoyer le texte extrait. Est-ce que ça aide ? Toujours ?

In [ ]:
# Votre code ici